# Notebook 06: Unsupervised K-Means Clustering
### Machine Learning Laboratory Mini-Project
**Objective**: Group laps into natural driving clusters without labels using $K$-Means ($K=3$) and visualize with 2-component PCA projection.

---

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

from backend.app.services.fastf1_service import FastF1DataService
from backend.app.services.feature_engineering import TelemetryFeatureExtractor

service = FastF1DataService()
df = service.load_session_data(season=2023, grand_prix='Monza')['features_df']
features = TelemetryFeatureExtractor.FEATURE_COLUMNS

X = df[features].values
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

k = 3
kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
clusters = kmeans.fit_predict(X_scaled)
df['cluster'] = clusters

# PCA Projection for 2D visualization
pca = PCA(n_components=2, random_state=42)
pca_coords = pca.fit_transform(X_scaled)
df['pc1'] = pca_coords[:, 0]
df['pc2'] = pca_coords[:, 1]

print(f'Explained Variance Ratio: {pca.explained_variance_ratio_}')

## 1. 2D PCA Cluster Scatter Plot

In [ ]:
plt.figure(figsize=(8, 6))
colors = ['#e10600', '#00d2be', '#ff8700']
for c in range(k):
    subset = df[df['cluster'] == c]
    plt.scatter(subset['pc1'], subset['pc2'], label=f'Cluster {c+1} (n={len(subset)})', color=colors[c], alpha=0.8)

plt.title('K-Means Driver Stint Clusters in 2D PCA Space')
plt.xlabel(f'Principal Component 1 ({pca.explained_variance_ratio_[0]*100:.1f}%)')
plt.ylabel(f'Principal Component 2 ({pca.explained_variance_ratio_[1]*100:.1f}%)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 2. Dynamic Cluster Centroid Profiles

In [ ]:
profile = df.groupby('cluster')[['avg_speed', 'full_throttle_pct', 'brake_pct', 'tyre_age', 'lap_time']].mean()
print('Cluster Mean Statistics:')
print(profile.round(2))

## 3. Academic Interpretation
- **Cluster 1**: Push Laps / Fresh Tyres (High speed, maximum throttle commitment).
- **Cluster 2**: Balanced Race Stint (Consistent pacing, controlled braking).
- **Cluster 3**: Heavy Tyre Degradation (Lower average speed, conservation driving).